In [ ]:
import os
import wandb

# [REDACTED - use environment variable, e.g. WANDB_API_KEY]
wandb.login()  # reads WANDB_API_KEY from environment

drive_output_path = os.environ.get('BONSAI_CKPT', '/home/ubuntu/Bonsai_MAX/checkpoints')
os.makedirs(drive_output_path, exist_ok=True)

print(f"Checkpoints se guardarán en: {drive_output_path}")


#Dependecy downloader

In [ ]:
%%capture

!pip install -q --upgrade pip uv

# ============================================
# 1. PyTorch stack FIJO
# torch 2.8.0 <-> torchvision 0.23.0
# ============================================

!pip install -q --force-reinstall \
    torch==2.8.0 \
    torchvision==0.23.0 \
    torchaudio==2.8.0 \
    --index-url https://download.pytorch.org/whl/cu128


# ============================================
# 2. xFormers compatible con PyTorch 2.8
# ============================================

!pip install -q \
    xformers==0.0.32.post2


# ============================================
# 3. Unsloth + Unsloth Zoo
# ============================================

!pip install -q \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"

!pip install -U bitsandbytes

# ============================================
# 4. Qwen3.5 necesita Transformers v5
# TRL que estás usando
# ============================================

!pip install -q \
    "transformers==5.2.0" \
    "trl==0.22.2"


# ============================================
# 5. Qwen3.5 linear attention
# ============================================

!pip install -q --no-build-isolation \
    flash-linear-attention \
    causal_conv1d==1.6.0

!pip install -q wandb


In [ ]:
!pip install -U bitsandbytes

##Check


In [ ]:
import torch
import torchvision
import transformers

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

import unsloth

print("🦥🦥🦥 Unsolth is ready 🦥🦥🦥")

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_FOR_THIS_TRAINING = "prism-ml/Bonsai-27B-unpacked" #Qwen 3.5 27B as a base

model , tokenizer_model = FastLanguageModel.from_pretrained(
    model_name = MODEL_FOR_THIS_TRAINING,
    max_seq_length = 8192,
    load_in_4bit = True, # 4 bit quantization to reduce memory
    load_in_8bit = False,
    full_finetuning = False,
)


In [ ]:
model = FastLanguageModel.get_peft_model(
   model,
   r= 32,
   target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
"gate_proj", "up_proj", "down_proj",],
   lora_alpha = 64,
   lora_dropout = 0,
   bias = "none",
   use_gradient_checkpointing = "unsloth",
   random_state = 3407,
   use_rslora = False,
   loftq_config = None,

)


##Data Pipeline


In [ ]:
from datasets import load_dataset,concatenate_datasets,Dataset
from unsloth.chat_templates import get_chat_template
import re
import json
import multiprocessing as theardL
import pandas as pandis
from transformers import AutoTokenizer

RANDOM_SEED = 12181531 #Reprodusibilty
MAX_CONTEXT_WINDOW = 8192

#SPLIT  TRAIN TEST VALIDATION
num_examples_data_set = {
    "qwen_38_data_set":5000,
    "qwen_38_code":10000,
    "swe_next":2000,
}

tokenizer = get_chat_template(
    tokenizer_model,
    chat_template="qwen3-thinking", #Important!!!!!
)

#I only need this two domain , to specify the model way to think
allowed_domains = {"code","reasoning"}

#Load the dataset from hf
def load_sample(dataset_name,sample_count=None,domains=None,split="train",subset=None):
  try:
    if subset:
      data_set = load_dataset(dataset_name,subset,split=split)
    else:
      data_set = load_dataset(dataset_name,split=split)

  except ValueError:
    raise

  if domains is not None:
    data_set = data_set.filter(
      lambda example: example["domain"] in domains
    ) #CODE REASONING

  if sample_count is not None:
    sample_count = min(sample_count,len(data_set))

    data_set = data_set.shuffle(seed=RANDOM_SEED).select(
      range(sample_count)
    )

  return data_set


#1 General reasoning + code dataset
qwen_38_data_set = load_sample(
    "r0b0tlab/qwen3.8-max-distillation-50k",
    num_examples_data_set["qwen_38_data_set"],
    split="train",
    domains=allowed_domains,
)


#2 Specialized code dataset
qwen_38_code = load_sample(
    "guell00/qwen-3.8-code",
    num_examples_data_set["qwen_38_code"],
    split="train",
)


#3 Software engineering / agent trajectories
swe_next = load_sample(
    "TIGER-Lab/SWE-Next-SFT-Trajectories",
    num_examples_data_set["swe_next"],
    split="train",
)

###Check dataset Arch to set it into the parser method


In [ ]:
print("QWEN 3.8")
print(qwen_38_data_set.column_names)
print(qwen_38_data_set[0])

print("\nQWEN CODE")
print(qwen_38_code.column_names)
print(qwen_38_code[0])

print("\nSWE NEXT")
print(swe_next.column_names)
print(swe_next[0])

###Formatting to unified conversations

In [ ]:
def _clean_strip(data):
  return (data or "").strip()
  #"    data  " -> "data" :)


THINK_BLOCK_RE = re.compile(
  r"<think>.*?</think>",
  flags=re.DOTALL
)


def normalize_assistant_to_think_solution(data_text: str) -> str:
  data_text = _clean_strip(data_text)

  if not data_text:
    return "<think></think>\n" #EMPTY THINKING BLOCK

  m = THINK_BLOCK_RE.search(data_text)

  if m:
    think_block = m.group(0).strip()
    rest = data_text[m.end():].lstrip()

    return (
      f"{think_block}\n{rest}".rstrip()
      if rest
      else f"{think_block}\n"
    )

  return f"<think></think>\n{data_text}".rstrip()


def build_assistant_with_reasoning(content: str, reasoning: str = "") -> str:
  content = _clean_strip(content)
  reasoning = _clean_strip(reasoning)

  if "<think>" in content and "</think>" in content:
    return normalize_assistant_to_think_solution(content)

  if reasoning:
    return (
      f"<think>{reasoning}</think>\n{content}"
      if content
      else f"<think>{reasoning}</think>\n"
    )

  return normalize_assistant_to_think_solution(content)


def parse_message_item(m):
  if isinstance(m, dict): #If is already a perfect dict
    return m

  if isinstance(m, str):
    s = _clean_strip(m)

    if not s:
      return None

    try:
      obj = json.loads(s)
      return obj if isinstance(obj, dict) else None

    except Exception:
      return None

  return None


##Now we create methods specific for each dataset to parse it

def format_qwen(examples):
  out = []

  for messages in examples["messages"]:
    cleaned = []

    for message in messages:
      role = message.get("role")
      content = _clean_strip(message.get("content",""))

      if role == "system":
        continue

      if role == "assistant":
        content = normalize_assistant_to_think_solution(content)

      if role in ("user","assistant"):
        cleaned.append({
          "role":role,
          "content":content
        })

    if len(cleaned) >= 2:
      out.append(cleaned)

  return {"conversations":out}


def format_swe(examples):
  out = []

  for messages in examples["messages"]:
    cleaned = []

    for message in messages:
      role = message.get("role")
      content = _clean_strip(message.get("content",""))

      if role == "system":
        continue

      if role in ("user","assistant","tool"):
        cleaned.append({
          "role":role,
          "content":content
        })

    if len(cleaned) >= 2:
      out.append(cleaned)

  return {"conversations":out}

#Now lets clean it :))

In [ ]:
qwen_38_data_set = qwen_38_data_set.map(
  format_qwen,
  batched=True,
  remove_columns=qwen_38_data_set.column_names
)

qwen_38_code = qwen_38_code.map(
  format_qwen,
  batched=True,
  remove_columns=qwen_38_code.column_names
)

swe_next = swe_next.map(
  format_swe,
  batched=True,
  remove_columns=swe_next.column_names
)

##Final dataset construction

In [ ]:
qwen_38_data_set = qwen_38_data_set.filter(
  lambda x: x["conversations"] is not None
  and len(x["conversations"]) > 0
)

qwen_38_code = qwen_38_code.filter(
  lambda x: x["conversations"] is not None
  and len(x["conversations"]) > 0
)

swe_next = swe_next.filter(
  lambda x: x["conversations"] is not None
  and len(x["conversations"]) > 0
)

combined_dataset = concatenate_datasets([
  qwen_38_data_set,
  qwen_38_code,
  swe_next
]).shuffle(seed=RANDOM_SEED)
##There we just mixed all the dataset in one with the same arch

def formatting_prompts_func(examples):
  convos = examples["conversations"]

  texts = [
    tokenizer.apply_chat_template(  ##Convert the data set into a QWEN STRUCT CHAT
      convo,
      tokenize=False,
      add_generation_prompt=False,
    )
    for convo in convos
  ]

  return {"text":texts}


dataset = combined_dataset.map(formatting_prompts_func, batched=True)

num_proc = theardL.cpu_count() ##JUST CPU CORE COUNT TO HUGGINFACE

_text_tok = getattr(tokenizer, "tokenizer", tokenizer)

#Just to not exceed ContextWindow
def filter_long_data(examples):
  texts = examples["text"]

  tokenized = _text_tok(
    texts,
    truncation=False,
    padding=False,
    add_special_tokens=False,
  )["input_ids"]

  return [
    len(toks) <= MAX_CONTEXT_WINDOW
    for toks in tokenized
  ]

dataset = dataset.filter( ##There we filter the dataset  to clean long data
  filter_long_data,
  batched=True,
  num_proc=num_proc
)


def check_assistant_format(examples):
  convos = examples["conversations"]
  ok = []

  for convo in convos:
    good = True

    for m in convo:
      if m["role"] == "assistant":
        c = m.get("content","")

        #SWE tool call -> valid
        if "<function=" in c:
          continue

        #Normal reasoning assistant -> must have think block
        if "<think>" not in c or "</think>" not in c:
          good = False
          break

        if not re.search(r"</think>\n",c):
          good = False
          break

    ok.append(good)

  return {"_ok":ok}


check = dataset.map(
  check_assistant_format,
  batched=True,
  remove_columns=dataset.column_names,
  num_proc=num_proc,
)
bad = len(check) - sum(check["_ok"])
if bad > 0:
  dataset = dataset.filter(
    lambda x: all(
      (m["role"] != "assistant")
      or ("<function=" in m.get("content",""))
      or (
        ("<think>" in m.get("content",""))
        and ("</think>\n" in m.get("content",""))
      )
      for m in x["conversations"]
    )
  )

print(dataset[0]["text"][:8000])

print("Final dataset:",len(dataset))

dataset_split = dataset.train_test_split(
  test_size=0.02,
  seed=RANDOM_SEED
)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print("Train:",len(train_dataset))
print("Eval:",len(eval_dataset))

##Trainer Configuration

In [ ]:
from trl import SFTTrainer, SFTConfig

pokemon_trainer = SFTTrainer(
  model=model, #Model to fine-tune
  tokenizer=tokenizer, #Tokenizer used for Qwen format
  train_dataset=train_dataset, #Training data
  eval_dataset=eval_dataset, #Validation data

  args=SFTConfig(
    dataset_text_field="text", #Column used for training
    max_length=MAX_CONTEXT_WINDOW, #Maximum context length

    per_device_train_batch_size=1, #Safe batch for 15GB VRAM
    per_device_eval_batch_size=1, #Safe eval batch
    gradient_accumulation_steps=16, #Simulates a larger batch

    num_train_epochs=1, #One full pass over the dataset
    learning_rate=1e-4, #LoRA learning speed
    warmup_ratio=0.03, #Gradually increases LR at start

    optim="adamw_8bit", #Memory-efficient optimizer
    weight_decay=0.001, #Small regularization
    lr_scheduler_type="linear", #Gradually reduces LR

    logging_steps=10,

    eval_strategy="steps", #Evaluate during training
    eval_steps=200, #Evaluate every 200 steps

    save_strategy="steps", #Save by training steps
    save_steps=200, #Save every 200 steps
    save_total_limit=1, #Keep only one checkpoint

    fp16=False,
    bf16=True,

    seed=RANDOM_SEED, #Reproducibility

    report_to="wandb", #Send metrics to Weights & Biases
    output_dir=drive_output_path, #Checkpoint output folder
    gradient_checkpointing_kwargs={"use_reentrant": False},
  ),
)


In [ ]:
from unsloth.chat_templates import train_on_responses_only

pokemon_trainer = train_on_responses_only(
  pokemon_trainer,
  instruction_part="<|im_start|>user\n",
  response_part="<|im_start|>assistant\n",
)

In [ ]:
tokenizer.decode([
  tokenizer.pad_token_id if x == -100 else x
  for x in pokemon_trainer.train_dataset[100]["labels"]
]).replace(tokenizer.pad_token," ")

##Lets Rock


In [ ]:
trainer_stats = pokemon_trainer .train()